# Remove null values and Deduplication

In [0]:
%sql
INSERT INTO silver.customers_source_crm
SELECT 
  CAST(_c0 AS INT) AS cst_id,
  _c1 AS cst_key,
  _c2 AS cst_firstname,
  _c3 AS cst_lastname,
  _c4 AS cst_marital_status,
  _c5 AS cst_gndr,
  CAST(_c6 AS DATE) AS cst_create_date
FROM csv.`/mnt/raw/source_crm/cust_info.csv`;

In [0]:
%sql
SELECT *
FROM silver.customers_source_crm limit 10;

In [0]:
CREATE OR REPLACE TABLE silver.customers_cleaned_crm
USING DELTA
AS
SELECT
  cst_id,
  cst_key,
  TRIM(cst_firstname) AS cst_firstname,
  TRIM(cst_lastname) AS cst_lastname,
  UPPER(TRIM(cst_marital_status)) AS cst_marital_status,
  UPPER(TRIM(cst_gndr)) AS cst_gndr,
  cst_create_date
FROM bronze.customers_source_crm
WHERE cst_id IS NOT NULL
  AND cst_create_date IS NOT NULL;



In [0]:
SELECT * FROM silver.customers_cleaned_crm LIMIT 10;

In [0]:
INSERT INTO silver.products_source_crm
SELECT
  CAST(_c0 AS INT) AS prd_id,
  _c1 AS prd_key,
  _c2 AS prd_nm,
  CAST(_c3 AS INT) AS prd_cost,
  _c4 AS prd_line,
  CAST(_c5 AS DATE) AS prd_start_dt,
  CAST(_c6 AS DATE) AS prd_end_dt
FROM csv.`/mnt/raw/source_crm/prd_info.csv`;


In [0]:
SELECT * FROM silver.products_source_crm Limit 10;

In [0]:
CREATE OR REPLACE TABLE silver.products_cleaned_crm
USING DELTA
AS
SELECT
  prd_id,
  TRIM(prd_key) AS prd_key,
  INITCAP(TRIM(prd_nm)) AS prd_nm,
  prd_cost,
  CASE
    WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
    WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
    WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'Other Sales'
    WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
    ELSE 'n/a'
  END AS prd_line_type,
  prd_start_dt,
  prd_end_dt
FROM bronze.products_source_crm
WHERE prd_id IS NOT NULL
  AND prd_cost IS NOT NULL;




In [0]:
SELECT * FROM silver.products_cleaned_crm LIMIT 10;

In [0]:
INSERT INTO silver.sales_source_crm
SELECT
  _c0 AS sls_ord_num,
  _c1 AS sls_prod_key,
  CAST(_c2 AS INT) AS sls_cust_id,
  CAST(_c3 AS DATE) AS sls_order_dt,
  CAST(_c4 AS DATE) AS sls_ship_dt,
  CAST(_c5 AS DATE) AS sls_due_dt,
  CAST(_c6 AS DOUBLE) AS sls_sales,
  CAST(_c7 AS INT) AS sls_quantity,
  CAST(_c8 AS DOUBLE) AS sls_price
FROM csv.`/mnt/raw/source_crm/sales_details.csv`;


In [0]:
SELECT * FROM silver.sales_source_crm LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE silver.sales_cleaned_crm
USING DELTA
AS
SELECT
  TRIM(sls_ord_num) AS sls_ord_num,
  TRIM(sls_prod_key) AS sls_prod_key,
  sls_cust_id,
  CAST(sls_order_dt AS DATE) AS sls_order_dt,
  CAST(sls_ship_dt AS DATE) AS sls_ship_dt,
  CAST(sls_due_dt AS DATE) AS sls_due_dt,
  CAST(sls_sales AS DOUBLE) AS sls_sales,
  sls_quantity,
  CAST(sls_price AS DOUBLE) AS sls_price
FROM bronze.sales_source_crm
WHERE sls_ord_num IS NOT NULL
  AND sls_cust_id IS NOT NULL
  AND sls_order_dt IS NOT NULL;



In [0]:
SELECT * FROM silver.sales_cleaned_crm;

In [0]:
INSERT INTO silver.CUST_AZ12_source_erp
SELECT
  CAST(_c0 AS INT) AS cid,
  CAST(_c1 AS DATE) AS bdate,
  _c2 AS gen
FROM csv.`/mnt/raw/source_erp/CUST_AZ12.csv`;

In [0]:
SELECT * FROM silver.CUST_AZ12_source_erp LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE silver.cust_cleaned_erp
USING DELTA
AS
SELECT
cid,
bdate,
 UPPER(TRIM(gen)) AS gen
FROM bronze.CUST_AZ12_source_erp;

In [0]:
SELECT * FROM silver.cust_cleaned_erp LIMIT 10;


In [0]:
INSERT INTO silver.LOC_A101_source_erp
SELECT
  _c0 AS cid    ,
  _c1 AS cntry  
FROM csv.`/mnt/raw/source_erp/LOC_A101.csv`;

In [0]:
CREATE OR REPLACE TABLE silver.location_cleaned_erp
USING DELTA
AS
SELECT
cid,
cntry
FROM bronze.LOC_A101_source_erp

In [0]:
SELECT * FROM silver.location_cleaned_erp LIMIT 10;

In [0]:
INSERT INTO silver.PX_CAT_G1V2_source_erp
SELECT
  _c0 AS id,
  _c1 AS cat,
  _c2 AS subcat,
  _c3 AS maintenance
FROM csv.`/mnt/raw/source_erp/PX_CAT_G1V2.csv`;

In [0]:
CREATE OR REPLACE TABLE silver.px_cat_cleaned_erp
USING DELTA
AS
SELECT
id,
cat,
subcat,
maintenance
FROM bronze.PX_CAT_G1V2_source_erp

In [0]:
SELECT * FROM silver.px_cat_cleaned_erp LIMIT 10;